# Tabela 19 — FT-CUR com seletor random (2026-10-14)

Refaz só as duas variantes corrigidas do FT-CUR (`FTCUR_minibatch`, `FTCUR_mfixed_full`) com `selection_method="random"`, o seletor do protocolo principal. Mesmo orçamento da rodada de 06/10 (200 épocas, paciência 16, 3 repetições). O resto da tabela (FT, SAINT) já é atual.

**Settings → Accelerator → GPU T4** antes de rodar. No fim, baixe `/kaggle/working/table19_ftcur_random.json`.

In [ ]:
# ── Célula 1: Verifica GPU ───────────────────────────────────────────
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memória: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    raise RuntimeError('GPU não detectada — ative em Settings > Accelerator')


In [ ]:
# ── 2. Repositório + verificação da instrumentação de orçamento ──
import os, subprocess, json, statistics as st
GIT_URL     = 'https://github.com/PauloBernardo/dissertacao-estudo-comparativo.git'
PROJECT_DIR = '/kaggle/working/sparse-lssvm-transformers-study'
BRANCH      = 'revisao/estatistica-e-proveniencia'
if os.path.exists(PROJECT_DIR):
    subprocess.run(['git', '-C', PROJECT_DIR, 'fetch', 'origin'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, GIT_URL, PROJECT_DIR], check=True)
subprocess.run(['git', '-C', PROJECT_DIR, 'checkout', BRANCH], check=True)
subprocess.run(['git', '-C', PROJECT_DIR, 'pull', '--rebase', 'origin', BRANCH], check=True)
os.chdir(PROJECT_DIR)
!git log --oneline -2
import sys; sys.path.insert(0, '.')

# A instrumentação tem de existir nos DOIS laços: o FT tem laço próprio, enquanto
# SAINT e FT-CUR passam por fit_model/train_epoch em ft_transformer_model.py.
import inspect
from src.models.transformers.ft_transformer import FTTransformer
assert 'self.best_epoch_' in inspect.getsource(FTTransformer.fit), 'FT sem best_epoch_ — clone velho, refaça o pull'
assert "'best_epoch'" in open('src/models/ft_transformer_model.py').read(), 'fit_model sem best_epoch'

import scripts.run_tier1_gridcv as _r1
assert hasattr(_r1, 'BUDGET_OVERRIDE') and hasattr(_r1, '_apply_budget'), 'runner sem override de orçamento'
# O override precisa acertar a chave de cada wrapper (FT usa max_epochs; SAINT/FT-CUR usam epochs).
_r1.BUDGET_OVERRIDE.update({'epochs': 200, 'patience': 16})
assert _r1._apply_budget({'max_epochs': 40, 'patience': 6}) == {'max_epochs': 200, 'patience': 16}
assert _r1._apply_budget({'epochs': 40, 'patience': 6})     == {'epochs': 200, 'patience': 16}
_r1.BUDGET_OVERRIDE.clear()
print('instrumentação OK: best_epoch/n_steps nos dois laços + override nas duas convenções de chave')

assert open('scripts/run_table19_benchmark.py').read().count('selection_method="random"') == 2, 'run_table19_benchmark sem o seletor random — faça o push e o pull'
print('FT-CUR do benchmark com seletor random: OK')


In [ ]:
# ── Célula 3: Dependências ─────────────────────────────────────────
!pip install -q entmax einops psutil
import torch, sklearn, numpy, psutil
print(f'torch {torch.__version__} | sklearn {sklearn.__version__} | numpy {numpy.__version__} | psutil {psutil.__version__}')


In [ ]:
# ── Rodar benchmark (só FT-CUR, uma GPU) ──
!CUDA_VISIBLE_DEVICES=0 python -u scripts/run_table19_benchmark.py \
    --variants FTCUR_minibatch,FTCUR_mfixed_full \
    --budget-epochs 200 --budget-patience 16 --repeats 3 \
    --output results/table19_ftcur_random.json \
    2>&1 | tee /kaggle/working/table19_ftcur_random.log

import shutil
shutil.copy('results/table19_ftcur_random.json', '/kaggle/working/table19_ftcur_random.json')
print('Salvo em /kaggle/working/table19_ftcur_random.json')


In [ ]:
# ── Resumo ──
import json
for r in json.load(open('results/table19_ftcur_random.json')):
    print(r['variant'], r['n'], 'skip' if r.get('skipped') else f"fit={r['fit_s_median']:.2f}s vram={r.get('vram_mb_median')} pred={r.get('pred_ms_median')}ms")
